# 5-2. Quantization — 모델 경량화와 추론 최적화

> **📌 이 노트북에 대하여**
>
> 4-1, 4-2, 5-1과 마찬가지로 **별도의 PPT 없이 이 노트북 하나로** 이론과 실습을 진행한다.
>
> 그리고 이번이 **정규 커리큘럼의 마지막 챕터**다.
> 지금까지 배운 것들이 여기서 어떻게 모이는지 함께 확인해 보자.

---

## 학습 목표

1. LLM의 **메모리 문제**와 양자화가 필요한 이유를 설명할 수 있다
2. **FP32 / FP16 / INT8 / INT4**의 차이를 이해하고, 양자화 수식을 직접 구현할 수 있다
3. **NF4**가 왜 LLM에 유리한지 **코드로 검증**할 수 있다
4. `BitsAndBytesConfig`로 INT4 양자화를 적용하고 **메모리·속도·품질을 측정**할 수 있다
5. 양자화의 **부작용**을 관찰하고, **프롬프트 강화**로 보완하는 전략을 설명할 수 있다

---

## 목차

| # | 내용 | 성격 |
|:---:|------|:---:|
| 0 | **환경 설정** — 패키지 설치, GPU 확인 | 실습 |
| 1 | **왜 Quantization이 필요한가** — LLM의 메모리 문제 | 이론 |
| 2 | **비트 정밀도의 이해** — FP32·FP16·INT8·INT4 + 양자화 수식 | 이론+실습 |
| 3 | **FP16 기준선 측정** — 메모리·속도의 비교 대상 확보 | 실습 |
| 4 | **INT4 양자화 적용** — bitsandbytes + NF4 | 이론+실습 |
| 5 | **양자화의 부작용** — 환경 변화 입력에서의 품질 저하 | 이론+실습 |
| 6 | **프롬프트로 품질 복구** — 추론 시점의 보완 전략 | 이론+실습 |
| 7 | **정리 + 전체 과정 회고** | — |

---

## 선행 지식 — 앞 챕터와의 연결

| 앞 챕터 | 이번 챕터에서 어떻게 쓰이는가 |
|---|---|
| **2-2** 프롬프팅 기법 | 6장의 품질 복구 전략이 **Role Prompting · Few-shot** 그 자체다 |
| **4-2** Tool-use | 양자화로 해결 안 되는 문제(산술)에 **도구를 붙이는** 대안 |
| **5-1** PEFT/QLoRA | ⭐ **`load_in_4bit=True`의 정체를 오늘 밝힌다** |

> **⭐ 지난 시간에 남겨둔 숙제**
>
> 5-1에서 QLoRA를 배울 때 이 한 줄을 썼다.
>
> ```python
> model, tokenizer = FastModel.from_pretrained(
>     'unsloth/gemma-3-1b-it-unsloth-bnb-4bit',
>     load_in_4bit=True,        # ← 이 옵션
> )
> ```
>
> 그때는 **"모델을 4-bit로 압축한다"** 는 수준으로만 넘어가고,
> **"원리는 5-2에서 배운다"** 고 했다.
>
> **오늘 그 안에서 무슨 일이 일어나는지 파헤친다.**

> **💡 이번 챕터의 한 줄 요약**
>
> 5-1(PEFT): **"모델을 어떻게 효율적으로 학습시킬 것인가"** — 학습 최적화
> 5-2(오늘): **"학습된 모델을 어떻게 가볍게 서빙할 것인가"** — 추론 최적화
>
> 👉 **학습과 배포는 한 세트다.** 오늘로 그 세트가 완성된다.


---

## 0. 환경 설정

### 이번 실습에 필요한 것

| 항목 | 내용 |
|---|---|
| **GPU** | CUDA 지원 GPU 필수 (bitsandbytes가 NVIDIA GPU에서만 동작) |
| VRAM | 최소 8GB (FP16 3.9GB + INT4 1.7GB를 순차 로딩) |
| 모델 | `Qwen/Qwen2.5-1.5B-Instruct` (약 3GB 다운로드) |

> **⚠️ 5-1과 달리 API 키가 필요 없다**
>
> 4-1, 4-2에서는 GMS API로 원격 모델을 호출했고,
> 5-1에서는 Unsloth로 모델을 받아 학습시켰다.
> 오늘은 **HuggingFace에서 모델을 직접 받아 내 GPU에서 추론**한다.
>
> ```
>    4-1·4-2  :  API 호출        (모델은 남의 서버에)
>    5-1      :  로컬 학습        (모델이 내 GPU에)
>    5-2      :  로컬 추론 최적화  (모델을 내 GPU에 '더 작게')
> ```


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 역할: 양자화 실습에 필요한 패키지 설치
#              (Docker 환경에서는 실행하지 않습니다)
#
# - transformers  : HuggingFace 모델 로딩 및 추론
# - bitsandbytes  : INT4/INT8 양자화 구현체 ★ 이번 실습의 핵심
#                   ⚠️ CUDA GPU 전용. CPU나 Apple Silicon에서는 동작하지 않는다
# - accelerate    : GPU 자동 분배 (device_map='auto'에 필요)
# - tokenizers    : 토크나이저 (2-1 챕터에서 다룬 그 라이브러리)
#
# 💡 5-1에서 쓴 Unsloth도 내부적으로 bitsandbytes를 사용한다.
#    즉 오늘 배우는 것이 5-1의 load_in_4bit=True 뒤에서 돌아가던 바로 그것이다.
# ═══════════════════════════════════════════════════════════
# %pip install transformers bitsandbytes torch accelerate tokenizers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import warnings
import logging

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: GPU 환경 확인 + 불필요한 경고 메시지 억제
#
# [Warning 해결]
# 1. triton not found: Windows 환경에서는 triton이 미지원.
#    flop counting(연산량 측정)에만 사용되므로 실습에 영향 없음.
# 2. FutureWarning (_check_is_size): bitsandbytes 내부 코드가
#    PyTorch 차기 버전 API를 아직 반영하지 않은 것. 동작에 영향 없음.
# → 두 경고 모두 실습 결과에 영향을 주지 않으므로 억제한다.
# ═══════════════════════════════════════════════════════════

# 불필요한 경고 억제
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*triton.*')
logging.getLogger('torch.utils.flop_counter').setLevel(logging.ERROR)

# GPU 확인
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1024**3:.1f} GB')
    print(f'CUDA 버전: {torch.version.cuda}')
else:
    print('⚠️ GPU를 사용할 수 없습니다. CUDA 환경을 확인하세요.')


---

## 1. 왜 Quantization이 필요한가?

![image_A](https://i.ibb.co/tP3HgBpG/image-A.png)

### 1-1. LLM의 근본적 문제: 크기

LLM의 성능은 **파라미터 수에 비례**하여 좋아지는 경향이 있다.
하지만 파라미터가 많아질수록 **메모리와 비용**도 비례하여 증가한다.

| 모델 | 파라미터 수 | FP16 메모리 | 필요 GPU |
|------|:---:|:---:|------|
| Qwen2.5-1.5B | 15억 개 | ~3GB | RTX 3060 (12GB) |
| Llama-3-8B | 80억 개 | ~16GB | RTX 4090 (24GB) |
| Llama-3-70B | 700억 개 | ~140GB | A100 × 2장 (160GB) |
| 초대형 모델 | 수천억 ~ 1조 (추정) | 수백 GB ~ TB급 | 수십 장의 고급 GPU |

> ⚠️ 상용 모델(GPT-4 등)의 정확한 파라미터 수는 **공개되지 않았다.**
> 위 표의 마지막 행은 업계 추정치이므로 참고만 하자.

우리가 사용하는 GPU의 VRAM은 **16GB**다.
FP16으로는 8B 모델도 빠듯하고, 13B 이상은 로딩조차 불가능하다.

**문제는 명확하다: 좋은 모델을 쓰고 싶지만, GPU 메모리가 부족하다.**

> **💡 5-1에서 이미 같은 벽을 만났다**
>
> 5-1 챕터 2에서 Full Fine-tuning 메모리를 계산했을 때를 떠올려 보자.
>
> ```
>    1B 모델 Full FT  ->  20GB 이상 필요  ->  16GB로는 불가능
> ```
>
> 그때는 **"학습"** 이 문제였고, 오늘은 **"추론"** 이 문제다.
> 그리고 **해결의 실마리도 같다 — 비트 수를 줄이는 것.**

### 1-2. Quantization(양자화)이란?

**Quantization = 모델의 가중치를 표현하는 비트 수를 줄이는 것**

> **💡 비유: 사진 압축**
>
> - **FP32** = RAW 사진 (원본 그대로, 파일 크기 큼)
> - **FP16** = JPEG 고화질 (거의 차이 없음, 50% 압축)
> - **INT4** = JPEG 저화질 (약간 흐릿하지만, 87.5% 압축)
>
> 핵심은 **"눈에 띄지 않을 정도의 손실로 최대한 압축"** 하는 것이다.

| 정밀도 | 비트 수 | 메모리 (1B 파라미터 기준) | FP32 대비 절감 |
|--------|:---:|:---:|:---:|
| FP32 | 32비트 | 4GB | 기준 |
| FP16 | 16비트 | 2GB | 50% |
| INT8 | 8비트 | 1GB | 75% |
| INT4 | 4비트 | 0.5GB | 87.5% |

INT4 양자화를 적용하면 Llama-3-70B(FP16: 140GB)를 **약 35GB**로 줄일 수 있다.
A100 1장(80GB)에서 충분히 돌릴 수 있게 되는 것이다.

### 1-3. Quantization의 트레이드오프

```
메모리/속도 ↑↑  vs  정확도 ↓
```

비트 수를 줄이면 메모리가 절감되지만, **정밀도 손실로 인한 품질 저하**가 발생할 수 있다.
특히 **노이즈가 있는 입력**(오타, 모호한 표현)에서 저하가 더 심해진다.
이 문제와 해결책을 챕터 5~6에서 다룬다.

> **⚠️ "양자화하면 무조건 빨라진다"는 오해**
>
> 메모리는 확실히 줄지만, **속도는 항상 빨라지지 않는다.**
> INT4로 저장한 가중치를 연산할 때마다 **FP16으로 되돌리는(dequantize) 비용**이 들기 때문이다.
>
> ```
>    소형 모델(1.5B)  ->  역양자화 오버헤드가 커서 오히려 느릴 수 있다
>    대형 모델(70B)   ->  메모리 절감 효과가 압도적이라 확실히 유리하다
> ```
>
> 👉 챕터 4에서 이것을 **직접 측정**해서 확인한다.

### 1-4. 5-1(PEFT)과의 관계 — 지난 시간 복습

지난 시간에 배운 QLoRA와 오늘 배울 PTQ는 **같은 양자화 기술을 다른 목적으로** 쓴다.

| 구분 | 5-1 QLoRA (지난 시간) | 5-2 PTQ (오늘) |
|------|:---:|:---:|
| 시점 | **학습** 시 | **추론** 시 |
| 목적 | 학습 메모리 절감 (적은 GPU로 튜닝) | 추론 메모리 절감, 서빙 비용 절약 |
| 양자화의 역할 | 베이스 모델을 **얼려두는 수단** | **주인공** |
| 학습 여부 | Adapter를 학습한다 | **학습하지 않는다** |
| 결과물 | 도메인 특화 모델 | 가볍게 서빙 가능한 모델 |

> **⭐ 실무에서는 이 둘이 이어진다**
>
> ```
>    ① 5-1 QLoRA로 도메인 특화 모델을 만든다
>              ↓
>    ② 5-2 PTQ로 그 모델을 가볍게 만들어 서빙한다
>
>    학습(5-1)  ->  배포(5-2)
> ```
>
> 오늘로 **"모델을 만들고 서비스하는 한 사이클"** 이 완성된다.


---

## 2. 비트 정밀도의 이해

![image_B](https://i.ibb.co/m5MDYJz9/image-B.png)

### 2-1. 숫자를 비트로 표현하는 방법

> **💡 먼저 비유부터 — 눈금이 몇 개인 자로 잴 것인가?**
>
> 길이를 재는 자를 떠올려 보자.
>
> ```
>    눈금 많은 자  :  |||||||||||||||||||||||||   -> 12.7cm 까지 정확히 잰다
>    눈금 적은 자  :  |     |     |     |         -> "대략 13cm" 정도만
> ```
>
> **비트 수 = 자의 눈금 개수**다.
> 눈금이 많으면 정밀하지만 **자가 무거워진다(= 메모리를 많이 쓴다).**
>
> 양자화는 **"눈금을 줄여서 자를 가볍게 만드는 것"** 이다.

컴퓨터는 모든 숫자를 `비트(0과 1)`로 표현한다.
**비트 1개는 스위치 하나**라고 생각하면 된다. 켜짐(1) 또는 꺼짐(0).

```
   비트 1개  ->  2가지 (0, 1)
   비트 2개  ->  4가지 (00, 01, 10, 11)
   비트 3개  ->  8가지
   비트 4개  ->  16가지        ← INT4 가 표현할 수 있는 단계 수
   비트 8개  ->  256가지       ← INT8
```

**비트가 1개 늘 때마다 표현할 수 있는 가짓수는 2배**가 된다.

```
FP32 (32비트): ████████████████████████████████  →  소수점 7~8자리 정밀도
FP16 (16비트): ████████████████                  →  소수점 3~4자리 정밀도
INT8 ( 8비트): ████████                          →  -128 ~ 127 정수 (256단계)
INT4 ( 4비트): ████                              →  -8 ~ 7 정수 (16단계)
```

> **⚠️ "16단계밖에 안 되는데 쓸모가 있나?"**
>
> 당연히 드는 의문이다. **바로 그 답이 2-2절과 2-4절에 있다.**
> 힌트: **LLM 가중치는 아무 값이나 나오는 게 아니라, 0 근처에 몰려 있다.**

### 2-2. 부동소수점 vs 정수

| 구분 | FP (Float) | INT (Integer) |
|------|:---:|:---:|
| 표현 방식 | 소수점이 "떠다니는" 실수 | 정수만 표현 |
| 예시 | 0.123456, -3.14 | 0, 3, -7 |
| 장점 | 넓은 범위, 높은 정밀도 | 연산 빠름, 메모리 적음 |
| 용도 | 학습, 원본 가중치 | **양자화된 가중치** |

LLM의 원본 가중치는 FP32/FP16(실수)이지만, 양자화하면 INT8/INT4(정수)로 변환된다.
추론할 때는 이 정수를 다시 실수로 복원(역양자화)하여 연산한다.

> **💡 핵심 질문: 16단계(INT4)만으로 LLM의 가중치를 표현할 수 있는가?**
>
> LLM의 가중치는 대부분 **0 근처에 밀집**된 정규분포를 따른다.
> 극단적으로 크거나 작은 값은 매우 드물다.
>
> ```
>    가중치 분포 (정규분포)
>
>            ▁▂▄███████▄▂▁
>       ─────┴──────┴──────┴─────
>          -0.06    0    0.06
>                   ↑
>            여기에 대부분이 몰려 있다
> ```
>
> 따라서 **0 근처를 더 세밀하게, 극단값은 거칠게** 표현해도 품질 손실이 크지 않다.
> 이것이 `NF4(Normal Float 4-bit)`의 핵심 아이디어이며,
> **챕터 2 마지막에서 코드로 직접 검증**한다.

### 2-3. 양자화는 어떻게 하는가

> **💡 비유: 키를 '학년'으로 바꾸기**
>
> 학생들의 키(실수)를 **1~5의 등급(정수)** 으로 바꾼다고 해보자.
>
> ```
>    1단계  가장 작은 키와 가장 큰 키를 찾는다      140cm ~ 190cm
>    2단계  그 범위를 5칸으로 나눈다                한 칸 = 10cm   ← 이게 scale
>    3단계  각 학생을 가까운 칸에 배정한다          163cm -> 3등급
>    4단계  나중에 등급으로 키를 되돌린다           3등급 -> 160cm
>                                                        ↑ 3cm 어긋남!
> ```
>
> **양자화가 정확히 이 과정이다.** 키 대신 가중치를, 등급 대신 정수를 쓸 뿐이다.
> 그리고 **4단계에서 생기는 어긋남이 곧 '양자화 오차'** 다.

#### 실제 수식

```
양자화:   Q(x) = round(x / scale) + zero_point
역양자화: x' = (Q(x) - zero_point) × scale
```

기호가 많아 보이지만 **하는 일은 위 비유 그대로**다.

| 기호 | 비유에서는 | 역할 |
|------|---|------|
| `x` | 학생의 실제 키 | 원본 가중치 (실수) |
| `scale` | **한 칸의 크기(10cm)** | 실수를 정수로 바꾸는 환산 비율 |
| `zero_point` | 기준점 맞추기 | 실수 0이 어느 정수에 오는지 |
| `Q(x)` | 배정된 등급 | 양자화된 정수 |
| `x'` | 등급으로 되돌린 키 | 복원된 실수 (**원본과 약간 다름**) |

> **📌 `zero_point`가 왜 필요한가?**
>
> 가중치에는 **음수도 있다.** 그런데 저장은 `0, 1, 2, ...` 같은 양수로 하고 싶다.
> 그래서 **전체를 양수 쪽으로 밀어주는 값**이 필요하다. 그게 `zero_point`다.
>
> ```
>    실제 값 :  -1.5  ...  0  ...  2.1
>                 ↓ zero_point 만큼 밀기
>    저장 값 :    0   ...  6  ...  15
> ```

양자화 → 역양자화 과정에서 `정보 손실(오차)`이 발생한다.
비트 수가 적을수록 오차가 커지고, 이것이 품질 저하의 원인이 된다.

> **⚠️ 왜 오차가 생기는가 — `round()` 때문이다**
>
> ```
>    원본 0.523  ->  0.523 / scale = 2.18  ->  round -> 2  ->  다시 곱하면 0.480
>                                                              ↑ 0.043 만큼 어긋남
> ```
>
> **소수를 정수로 바꾸는 순간 반올림이 일어나고, 그 차이가 곧 오차다.**
> 단계(표현 가능한 정수의 개수)가 많을수록 반올림 폭이 작아진다.
>
> | | 단계 수 | 반올림 폭 |
> |---|:---:|:---:|
> | INT8 | 256 | 좁다 → 오차 작음 |
> | INT4 | 16 | **넓다 → 오차 큼** |

아래에서 이 과정을 직접 코드로 체험해 보자.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: FP32 vs FP16의 정밀도 차이를 직접 확인
#
# [왜 이 실험을 하는가?]
# FP32(32비트)와 FP16(16비트)은 같은 숫자를 다른 정밀도로 표현한다.
# FP16은 메모리를 50% 절감하는 대신 소수점 아래 정밀도가 떨어진다.
# 하지만 LLM의 가중치에서 이 정도 오차는 무시할 수 있는 수준이다.
# 그래서 현재 대부분의 LLM 추론은 FP16을 기본 정밀도로 사용한다.
#
# [예상 결과]
# - 원주율(3.14...): FP16도 3.14까지는 정확 → 오차 매우 작음
# - 아주 작은 수(0.00001): FP16에서 약간의 오차 발생
# - FP16 최대값(65504): 정확히 표현 가능 (FP16의 최대 표현 범위)
# - 일반 소수(0.123...): FP16도 0.123까지는 정확
# ═══════════════════════════════════════════════════════════

test_values = [
    3.141592653589793,   # 원주율 — 소수점 몇 자리까지 정확한가?
    0.00001,             # 아주 작은 수 — FP16에서 표현 가능한가?
    65504.0,             # FP16 최대값 근처 — 오버플로우는 없는가?
    0.123456789,         # 일반적인 LLM 가중치 크기 — 실제 사용 범위
]

print('FP32 vs FP16 숫자 표현 비교')
print('=' * 60)
for value in test_values:
    # torch.tensor()로 같은 숫자를 다른 정밀도로 생성
    fp32 = torch.tensor(value, dtype=torch.float32)  # 32비트 실수
    fp16 = torch.tensor(value, dtype=torch.float16)  # 16비트 실수
    diff = abs(fp32.item() - fp16.item())  # 오차 = |FP32 - FP16|
    print(f'원본:  {value}')
    print(f'FP32:  {fp32.item():.15f}')
    print(f'FP16:  {fp16.item():.15f}')
    print(f'오차:  {diff:.15f}')
    print()

# [결과 해석]
# FP16의 오차는 10^-4 ~ 10^-8 수준으로, LLM 가중치 표현에는 충분하다.
# 이것이 FP16이 추론의 기본 정밀도로 사용되는 이유이다.
# 하지만 INT4(16단계)는 이보다 훨씬 거친 표현이므로 오차가 더 커진다.
print('→ FP32와 FP16의 차이는 매우 작다 (10^-4 이하).')
print('  이것이 FP16이 추론의 기본 정밀도로 사용되는 이유이다.')
print('  그렇다면 INT4(16단계)에서는 오차가 얼마나 커질까? → 다음 셀에서 확인')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 양자화 수식을 직접 코드로 실행하여 체험
#
# [양자화란?]
# 실수(0.5, -1.2 등)를 정수(3, -2 등)로 변환하는 과정이다.
# 변환 과정에서 "반올림"이 발생하므로 원본과 약간의 오차가 생긴다.
# 비트 수가 적을수록(INT4 < INT8) 표현 가능한 단계가 줄어 오차가 커진다.
#
# [수식]
# 양자화:   Q(x) = round(x / scale) + zero_point
# 역양자화: x' = (Q(x) - zero_point) × scale
#
# scale = 데이터 범위를 정수 범위에 매핑하는 "축척 비율"
# zero_point = 실수 0이 매핑되는 정수 값
#
# [예상 결과]
# INT8(256단계): 평균 오차 ~0.0035 (거의 원본과 동일)
# INT4(16단계) : 평균 오차 ~0.040  → INT8의 약 12배
#
# ⚠️ 오차의 '절대값'만 보면 0.04는 작아 보인다.
#    하지만 이 오차가 15억 개 파라미터에 걸쳐 누적되고,
#    수십 개 층을 통과하며 증폭된다는 점이 문제다. (챕터 5)
#
# 💡 clamp()가 INT4에만 있는 이유
#    이론상 Q값은 0~levels 범위에 들어가지만,
#    부동소수점 반올림 때문에 경계에서 벗어날 수 있다.
#    범위를 벗어나면 역양자화 시 엉뚱한 값이 되므로 안전장치를 둔다.
# ═══════════════════════════════════════════════════════════

# 1. 원본 텐서: LLM 가중치 5개를 시뮬레이션
# 실제 LLM은 수십억 개의 가중치를 갖지만, 원리는 동일하다.
original = torch.tensor([0.5, 1.2, -0.3, 2.1, -1.5], dtype=torch.float16)
print(f'1. 원본 텐서 (FP16): {original.tolist()}')

# 2. 데이터 범위 분석: 양자화의 scale을 계산하기 위한 준비
# scale = (최대값 - 최소값) / (표현 가능한 단계 수)
x_min, x_max = original.min().item(), original.max().item()
print(f'2. 데이터 범위: [{x_min:.4f}, {x_max:.4f}]')

# ========== INT8 양자화 (8비트 = 256단계) ==========
# 256단계로 나누므로 scale이 매우 작다 → 정밀한 표현 가능
levels_int8 = 2**8 - 1  # 255 (0~255까지 256개 값)
scale_int8 = (x_max - x_min) / levels_int8  # 각 단계의 간격
zero_point_int8 = round(-x_min / scale_int8) # 0이 매핑되는 정수

# ★ 핵심 수식: Q(x) = round(x / scale) + zero_point
quantized_int8 = torch.round(original / scale_int8) + zero_point_int8
# 역양자화: 정수를 다시 실수로 복원 (이때 오차가 발생한다)
dequantized_int8 = (quantized_int8 - zero_point_int8) * scale_int8

print(f'\n3. INT8 양자화 (256단계):')
print(f'   scale = {scale_int8:.6f} (각 단계의 간격)')
print(f'   양자화 값:  {quantized_int8.tolist()} ← 정수로 변환됨')
print(f'   복원 값:    {[round(v, 4) for v in dequantized_int8.tolist()]} ← 원본과 비교')
error_int8 = (original.float() - dequantized_int8.float()).abs().mean().item()
print(f'   평균 오차:  {error_int8:.6f} ← 256단계이므로 오차가 매우 작다')

# ========== INT4 양자화 (4비트 = 16단계) ==========
# 16단계로 나누므로 scale이 크다 → 거친 표현, 더 큰 오차
levels_int4 = 2**4 - 1  # 15 (0~15까지 16개 값)
scale_int4 = (x_max - x_min) / levels_int4  # INT8보다 ~17배 큰 간격
zero_point_int4 = round(-x_min / scale_int4)

quantized_int4 = torch.round(original / scale_int4) + zero_point_int4
quantized_int4 = quantized_int4.clamp(0, levels_int4)  # 0~15 범위로 제한
dequantized_int4 = (quantized_int4 - zero_point_int4) * scale_int4

print(f'\n4. INT4 양자화 (16단계):')
print(f'   scale = {scale_int4:.6f} (INT8의 {scale_int4/scale_int8:.0f}배 → 더 거친 표현)')
print(f'   양자화 값:  {quantized_int4.tolist()} ← 0~15 사이 정수')
print(f'   복원 값:    {[round(v, 4) for v in dequantized_int4.tolist()]} ← 원본과 차이 발생')
error_int4 = (original.float() - dequantized_int4.float()).abs().mean().item()
print(f'   평균 오차:  {error_int4:.6f} ← INT8보다 {error_int4/max(error_int8,1e-10):.0f}배 큰 오차')

# [결과 해석]
# INT4의 오차가 INT8보다 크지만, 이 정도는 LLM에서 허용 가능한 수준이다.
# 다만 이 "작은 오차"가 수십억 개 파라미터에 걸쳐 누적되면
# 특히 노이즈/오타 같은 비정상 입력에서 품질 저하로 이어진다. (챕터 5)
print(f'\n→ INT4 오차가 INT8보다 크지만, 단일 가중치 수준에서는 허용 범위이다.')
print(f'  문제는 이 오차가 15억 개 파라미터에 걸쳐 "누적"된다는 것이다. (챕터 5에서 확인)')
print(f'\n💡 그런데 여기서 의문이 생긴다.')
print(f'   "16단계를 균등하게 나누는 게 최선일까?"  -> 다음 셀에서 확인')


### 2-4. ⭐ 균등 분할이 최선인가? — NF4의 아이디어를 검증한다

앞 셀의 INT4는 **16단계를 균등한 간격으로** 배분했다.
그런데 LLM 가중치는 **0 근처에 몰려 있는 정규분포**다.

```
   [균등 분할]  값이 거의 없는 바깥 구간에도 단계를 똑같이 낭비한다
     ├────┼────┼────┼────┼────┼────┼────┼────┤
    -0.1              0                    0.1
                 ▁▂▄███▄▂▁
                 ↑ 실제 데이터는 여기 몰려 있다

   [분포 기반]  데이터가 많은 곳에 단계를 더 많이 배치한다
     ├────────┼──┼─┼┼┼┼┼─┼──┼────────┤
    -0.1              0                    0.1
```

**같은 4비트를 쓰면서, 단계를 어디에 배치하느냐만 바꾸는 것**이다.
이것이 `NF4(Normal Float 4-bit)`의 핵심 아이디어다.

> **📌 아래 코드는 NF4 그 자체가 아니라 '아이디어를 검증하는 실험'이다**
>
> 실제 NF4는 정규분포의 이론적 분위수로 만든 **고정 상수표**를 사용하고,
> 블록 단위 정규화 등 추가 기법이 들어간다.
> 여기서는 **"분포를 고려하면 같은 비트로도 오차가 줄어든다"** 는
> 핵심 원리만 확인한다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: "같은 4비트인데 배치만 바꾸면 오차가 줄어드는가?"
#
# [실험 설계]
# LLM 가중치를 흉내낸 정규분포 샘플 10만 개를 만들고,
# 두 가지 4비트 양자화를 적용해 평균 오차를 비교한다.
#
#   ① 균등 분할     : 최소~최대를 16등분 (앞 셀에서 한 방식)
#   ② 분포 기반     : 데이터의 분위수로 16개 대표값 선정 (NF4 아이디어)
#
# [핵심 관찰]
# - 두 방식 모두 '4비트(16단계)'를 쓴다  -> 메모리 사용량 동일
# - 그런데 오차는 다르다                 -> 배치 전략의 차이
# ═══════════════════════════════════════════════════════════

torch.manual_seed(42)  # 재현성 고정

# LLM 가중치를 흉내낸 정규분포 샘플 (실제 가중치도 이런 분포를 따른다)
weights = torch.randn(100_000) * 0.02

def quantize_uniform(x, bits=4):
    """① 균등 분할: 최소~최대를 2^bits 단계로 '똑같은 간격'으로 나눈다."""
    levels = 2**bits - 1
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / levels
    q = torch.round((x - x_min) / scale).clamp(0, levels)   # 양자화
    return q * scale + x_min                                 # 역양자화

def quantize_by_distribution(x, bits=4):
    """② 분포 기반: 데이터의 분위수로 대표값을 정한다 (NF4 아이디어).

    torch.quantile()로 0%, 6.7%, 13.3% ... 100% 지점의 값을 뽑으면
    데이터가 몰린 구간(0 근처)에서 대표값이 촘촘해진다.
    """
    levels = 2**bits                       # 16개 대표값
    qs = torch.linspace(0, 1, levels)
    codebook = torch.quantile(x, qs)       # ★ 분포를 반영한 대표값 표
    # 각 값을 가장 가까운 대표값으로 매핑 (경계 = 이웃 대표값의 중점)
    boundaries = (codebook[1:] + codebook[:-1]) / 2
    idx = torch.bucketize(x, boundaries)
    return codebook[idx.clamp(0, levels - 1)]

# ========== 두 방식의 오차 비교 ==========
err_uniform = (weights - quantize_uniform(weights)).abs().mean().item()
err_dist    = (weights - quantize_by_distribution(weights)).abs().mean().item()

print('같은 4비트(16단계)로 정규분포 가중치 10만 개를 양자화')
print('=' * 60)
print(f'① 균등 분할        평균 오차: {err_uniform:.6f}')
print(f'② 분포 기반(NF4식)  평균 오차: {err_dist:.6f}')
print('=' * 60)
print(f'→ 분포를 고려하면 오차가 {err_uniform / err_dist:.2f}배 작아진다.')
print('  메모리는 똑같이 4비트인데도!')

# ========== 대표값이 실제로 촘촘해졌는지 확인 ==========
codebook = torch.quantile(weights, torch.linspace(0, 1, 16))
gaps = (codebook[1:] - codebook[:-1]).tolist()

print('\n대표값 16개 사이의 간격:')
print(f'  0 근처 (가운데)  : {gaps[7]:.4f}   ← 촘촘하다')
print(f'  바깥쪽 (양 끝)   : {gaps[0]:.4f}   ← 성기다')
print(f'  차이            : 약 {gaps[0] / gaps[7]:.0f}배')
print('\n→ 데이터가 몰린 0 근처에 단계를 더 많이 배치했다는 뜻이다.')
print('  이것이 NF4가 같은 4비트로도 품질 손실이 적은 이유다.')

> **🔍 이 결과가 말해주는 것**
>
> | 관점 | 확인한 것 |
> |---|---|
> | **메모리** | 두 방식 모두 4비트 → **완전히 동일** |
> | **오차** | 분포 기반이 약 1.4배 작음 |
> | **차이의 원인** | 단계를 **어디에 배치했는가** 하나뿐 |
>
> ⭐ **"공짜 점심"에 가까운 개선이다.**
> 추가 비용 없이 배치 전략만 바꿔서 품질을 올렸다.
>
> 💡 **왜 이런 발상이 가능했나?**
> **데이터의 분포를 알고 있었기 때문**이다.
> LLM 가중치가 정규분포를 따른다는 사실을 알았기에 거기에 맞춰 최적화할 수 있었다.
>
> 👉 1-1 챕터의 **EDA 정신**이 여기서도 통한다.
> **데이터를 먼저 이해해야 좋은 방법이 나온다.**

> **📌 5-1의 그 옵션이 바로 이것이었다**
>
> ```python
> # 5-1에서 썼던 코드
> load_in_4bit=True
>
> # 오늘 챕터 4에서 직접 지정할 코드
> BitsAndBytesConfig(
>     load_in_4bit=True,
>     bnb_4bit_quant_type='nf4',   # ← 방금 검증한 그 아이디어
> )
> ```
>
> 5-1에서 Unsloth가 **자동으로 켜준 옵션**의 정체가 이것이다.


---

## 3. FP16 모델 로딩과 측정 — 기준선 확인

### 3-1. 왜 기준선(Baseline)이 필요한가?

양자화의 효과를 제대로 평가하려면 **비교 대상**이 필요하다.
"INT4 모델의 메모리가 1.7GB"라고만 해서는 좋은 건지 알 수 없다.
"FP16이 3.9GB인데 INT4가 1.7GB"라고 해야 **56% 절감**이라는 의미가 전달된다.

이 챕터에서 측정할 기준선:

| 지표 | 설명 | 왜 중요한가 |
|------|------|------|
| **모델 메모리** | 가중치가 차지하는 GPU 메모리 | 양자화의 직접적 절감 효과 |
| **피크 메모리** | 추론 중 최대 GPU 메모리 | 실제 운영에서의 VRAM 한도 |
| **추론 시간 (Latency)** | 응답 생성까지 걸리는 시간 | 서비스 응답 속도 |
| **응답 품질** | 생성된 텍스트의 정확성 | 양자화로 인한 품질 손실 여부 |

### 3-2. 이 실습에서 사용하는 모델

**`Qwen/Qwen2.5-1.5B-Instruct`**

| 항목 | 값 |
|------|------|
| 파라미터 수 | 약 15억 개 |
| FP16 예상 메모리 | 약 3~4GB |
| 선택 이유 | 16GB VRAM에서 FP16/INT4 **모두** 로딩 가능, 비교 실험에 적합 |

> **📌 왜 5-1의 Gemma가 아니라 Qwen인가?**
>
> 5-1에서 쓴 `unsloth/gemma-3-1b-...-bnb-4bit`는 **이미 4-bit로 양자화된 모델**이다.
> 오늘은 **양자화 전후를 비교**해야 하므로, 원본(FP16) 상태의 모델이 필요하다.
>
> ```
>    5-1 : 처음부터 4-bit 모델을 받아서 학습     (비교 불필요)
>    5-2 : FP16 원본을 받아 -> 직접 4-bit로 변환  (비교가 목적)
> ```
>
> Qwen2.5-1.5B는 FP16 원본이 공개되어 있고 크기도 적당해서 비교 실험에 알맞다.

### 3-3. HuggingFace에서 모델 로딩하기

HuggingFace의 `AutoModelForCausalLM.from_pretrained()`로 모델을 로딩한다.

핵심 파라미터:

```python
model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-1.5B-Instruct',
    dtype=torch.float16,  # 정밀도 지정 (FP16)
    device_map='auto',          # GPU에 자동 배치
)
```

- `dtype`: 모델 가중치의 정밀도. `torch.float16`이면 FP16으로 로딩
- `device_map='auto'`: GPU 메모리를 고려하여 자동 배치 (`accelerate` 라이브러리 필요)

> **⚠️ 측정할 때 반드시 지켜야 할 것 — 공정한 비교**
>
> FP16과 INT4를 비교하려면 **다른 조건은 전부 같아야** 한다.
>
> | 통제할 것 | 방법 |
> |---|---|
> | 같은 질문 | `test_prompt` 변수를 재사용 |
> | 같은 생성 방식 | `do_sample=False` (랜덤성 제거) |
> | 같은 GPU 상태 | INT4 로딩 전에 **FP16을 완전히 해제** |
>
> ⭐ 마지막 항목이 특히 중요하다. FP16이 GPU에 남아 있으면
> INT4 메모리 측정값에 **두 모델이 합산**되어 엉뚱한 결과가 나온다.
> 챕터 4 코드에서 `del model_fp16` 을 하는 이유가 이것이다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: FP16 모델 로딩 + 모델 메모리 측정
#
# [핵심 파라미터]
# - dtype=torch.float16: 가중치를 16비트 부동소수점으로 로딩
#   (32비트 대비 메모리 50% 절감, 품질 차이 거의 없음)
# - device_map='auto': GPU 메모리를 고려하여 자동 배치
#   (accelerate 라이브러리가 내부적으로 처리)
#
# [메모리 측정 방법]
# torch.cuda.memory_allocated(): 현재 GPU에 할당된 메모리
# → 모델 로딩 직후 측정하면 = 가중치(weights)만의 메모리
# ═══════════════════════════════════════════════════════════

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'

print('FP16 모델 로딩 중...')
torch.cuda.reset_peak_memory_stats()  # 메모리 통계 초기화

# ★ 핵심: dtype=torch.float16으로 16비트 정밀도 지정
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # FP16 정밀도
    device_map='auto',          # GPU에 자동 배치
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델 로딩 직후 메모리 = 가중치만의 크기
model_memory_fp16 = torch.cuda.memory_allocated() / 1024**3
print(f'모델 로딩 완료: {model_name}')
print(f'FP16 모델 메모리: {model_memory_fp16:.2f} GB')

# [예상 결과]
# Qwen2.5-1.5B = 약 15억 파라미터 × 2바이트(FP16) ≈ 3GB
# 실제로는 임베딩 테이블 등의 오버헤드로 ~3.9GB 정도 나온다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 파라미터 수 → 이론적 메모리 계산
#
# [계산 공식]
# 메모리(GB) = 파라미터 수 × 바이트/파라미터 ÷ (1024^3)
# - FP32: 1개 파라미터 = 4바이트 (32비트 ÷ 8)
# - FP16: 1개 파라미터 = 2바이트 (16비트 ÷ 8)
# - INT8: 1개 파라미터 = 1바이트 ( 8비트 ÷ 8)
# - INT4: 1개 파라미터 = 0.5바이트 (4비트 ÷ 8)
#
# [왜 이론값과 실측값이 다른가?]
# 이론값은 "순수 가중치"만 계산한다.
# 실측값에는 모델 구조 메타데이터, 임베딩 테이블, 버퍼 등
# 추가 오버헤드가 포함되어 이론값보다 약간 크게 나온다.
# ═══════════════════════════════════════════════════════════

# p.numel(): 하나의 파라미터 텐서에 포함된 숫자(원소)의 개수
total_params = sum(p.numel() for p in model_fp16.parameters())

# 정밀도별 이론적 메모리 계산
memory_fp32 = total_params * 4 / 1024**3   # 32비트 = 4바이트/파라미터
memory_fp16_theory = total_params * 2 / 1024**3   # 16비트 = 2바이트
memory_int8 = total_params * 1 / 1024**3   # 8비트 = 1바이트
memory_int4 = total_params * 0.5 / 1024**3 # 4비트 = 0.5바이트

print(f'총 파라미터 수: {total_params:,}개 ({total_params/1e9:.2f}B)')
print(f'\n정밀도별 이론적 메모리 (순수 가중치만):')
print(f'  FP32: {memory_fp32:.2f} GB')
print(f'  FP16: {memory_fp16_theory:.2f} GB (FP32 대비 50% 절감)')
print(f'  INT8: {memory_int8:.2f} GB (FP32 대비 75% 절감)')
print(f'  INT4: {memory_int4:.2f} GB (FP32 대비 87.5% 절감)')

print(f'\n실측값: {model_memory_fp16:.2f} GB (이론값 {memory_fp16_theory:.2f}GB + 오버헤드)')
print(f'→ 같은 모델이 INT4에서는 FP16의 약 1/4 메모리만 사용한다.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 추론 측정 함수 정의 + FP16 기준선 추론
#
# [measure_inference 함수]
# 모델에 질문을 보내고 3가지를 측정한다:
# 1. response: 생성된 텍스트 (응답 품질 확인용)
# 2. latency: 추론에 걸린 시간 (초)
# 3. peak_memory: 추론 중 최대 GPU 메모리 (GB)
#
# [apply_chat_template]
# Qwen 같은 Instruct 모델은 특정 대화 형식을 기대한다.
# apply_chat_template()이 이 형식으로 자동 변환해 준다.
# 예: {role: 'user', content: '질문'} → '<|im_start|>user\n질문<|im_end|>'
#
# [do_sample=False]
# 랜덤 샘플링을 끄고 항상 가장 확률 높은 토큰을 선택한다.
# FP16과 INT4 비교 시 "같은 조건"을 보장하기 위한 설정이다.
# ═══════════════════════════════════════════════════════════

def measure_inference(model, tokenizer, prompt, max_new_tokens=128):
    """모델 추론을 수행하고 latency와 peak memory를 측정한다."""
    torch.cuda.synchronize()  # GPU 작업 완료 대기
    torch.cuda.reset_peak_memory_stats()  # 피크 메모리 통계 초기화

    # 입력 준비: chat template 적용
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors='pt',
        return_dict=True, add_generation_prompt=True
    ).to(model.device)

    # 추론 실행 + 시간 측정
    start = time.time()
    with torch.no_grad():  # 그래디언트 계산 비활성화 (추론 시 불필요)
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False  # 결정적 출력 (비교 일관성)
        )
    torch.cuda.synchronize()  # GPU 작업 완료 대기
    latency = time.time() - start

    # 응답 디코딩: 입력 부분을 제외하고 새로 생성된 토큰만 추출
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],  # [입력길이:] = 새 토큰만
        skip_special_tokens=True
    )

    # 피크 메모리: 추론 과정에서 GPU 메모리가 가장 높았던 순간의 값
    # (가중치 + 활성화값 + KV 캐시 + 임시 버퍼 포함)
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
    return response, latency, peak_memory

# ========== FP16 기준선 추론 ==========
test_prompt = '서울에서 부산까지 KTX로 얼마나 걸리나요?'
response_fp16, latency_fp16, peak_fp16 = measure_inference(
    model_fp16, tokenizer, test_prompt
)

print(f'질문: {test_prompt}')
print(f'응답: {response_fp16[:200]}')
print(f'\n추론 시간: {latency_fp16:.2f}초')
print(f'모델 메모리: {model_memory_fp16:.2f} GB (가중치만)')
print(f'피크 메모리: {peak_fp16:.2f} GB (추론 중 최대 = 가중치 + 활성화 + KV캐시)')

# [결과 해석]
# 모델 메모리 < 피크 메모리: 추론 중 활성화값과 KV 캐시가 추가로 생성되기 때문
# 1.5B 모델에서는 이 차이가 작지만, 7B+ 모델에서는 차이가 커진다.


---

## 4. INT4 양자화 적용 — bitsandbytes + NF4

![image_C](https://i.ibb.co/d03m8fXF/image-C.png)

### 4-1. PTQ (Post-Training Quantization)

> **💡 비유: 이사와 짐 줄이기**
>
> ```
>    PTQ  :  이사 다 하고 나서 짐을 줄인다
>            -> 간단하다. 대신 버리다 보면 필요한 게 섞여 나갈 수 있다
>
>    QAT  :  이사 갈 집 크기를 '미리 알고' 짐을 싸면서 정리한다
>            -> 손실이 적다. 대신 처음부터 다시 싸야 해서 번거롭다
> ```
>
> **PTQ는 "완성된 모델을 나중에 압축", QAT는 "압축될 걸 알고 학습"** 이다.

양자화를 적용하는 시점에 따라 크게 두 가지 방식이 있다.

| 구분 | PTQ (이번 실습) | QAT (참고) |
|------|:---:|:---:|
| 이름 | Post-Training Quantization | Quantization-Aware Training |
| 시점 | **학습 완료 후** | 학습 중 |
| 원리 | 완성된 모델의 가중치를 정수로 변환 | 학습 과정에서 양자화 오차를 함께 학습 |
| 재학습 | **불필요** | 필요 |
| 적용 난이도 | **낮음** (코드 1~2줄) | 높음 (학습 파이프라인 수정) |
| 품질 | 약간 손실 가능 | 손실 최소화 |

<br>

이 실습에서는 **PTQ**를 사용한다. 이미 학습된 Qwen2.5-1.5B 모델에 양자화를 "사후 적용"하는 것이다.
코드 한 줄(`quantization_config=...`)만 추가하면 되므로 가장 간편하다.

> **💡 그럼 5-1의 QLoRA는 PTQ인가 QAT인가?**
>
> **엄밀히는 둘 다 아니다.** QLoRA는 제3의 방식이다.
>
> ```
>    PTQ   :  학습 끝난 모델을 양자화.  이후 학습 없음
>    QAT   :  양자화를 '고려하며' 모델 전체를 학습
>    QLoRA :  양자화된 모델은 '얼려두고', 별도 Adapter만 학습   ← 5-1
> ```
>
> QLoRA에서 **양자화된 가중치 자체는 절대 학습되지 않는다.**
> 학습되는 것은 16-bit로 유지되는 LoRA Adapter뿐이다.
> 5-1에서 "압축된 것은 얼려두고, 새로 붙인 것만 정밀하게 학습한다"고 했던 그 구조다.

### 4-2. NF4 (Normal Float 4-bit) — LLM에 최적화된 양자화

일반 INT4는 -8~7의 16단계를 **균등하게** 배분한다.

하지만 LLM의 가중치는 0 근처에 밀집된 **정규분포**를 따른다.

```
[일반 INT4]        균등 분포 (모든 구간 동일 간격)
  ├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
 -8                    0                     7

[NF4]              정규분포 최적화 (0 근처 더 세밀)
  ├────────┼──┼─┼┼┼┼┼─┼──┼────────┤
 -8           0 (촘촘)              7
```

NF4는 0 근처 구간을 더 조밀하게 배분하여 LLM 가중치의 분포에 최적화되어 있다.
일반 INT4보다 **같은 4비트에서도 품질 손실이 적다**.

> ⭐ **챕터 2-4에서 이미 코드로 검증했다.**
> 같은 4비트인데 대표값을 분포에 맞춰 배치하니 오차가 약 1.4배 줄었다.
> `bnb_4bit_quant_type='nf4'`가 바로 그 아이디어를 구현한 것이다.

### 4-3. BitsAndBytesConfig — 옵션 4개를 순서대로 이해하기

bitsandbytes는 HuggingFace와 연동되는 양자화 라이브러리이다.
설정 옵션이 4개인데, **"왜 필요한가"를 하나씩 따라가면** 쉽게 이해된다.

```python
BitsAndBytesConfig(
    load_in_4bit=True,                    # ① 4비트로 압축할래
    bnb_4bit_quant_type='nf4',            # ② 압축 방식은 NF4로
    bnb_4bit_compute_dtype=torch.float16, # ③ 계산할 땐 FP16으로 풀어서
    bnb_4bit_use_double_quant=True,       # ④ 남는 것도 마저 압축
)
```

**① `load_in_4bit=True` — "4비트로 압축할래"**

가장 기본. 이 한 줄이 양자화의 스위치다.
(5-1에서 Unsloth에 넘겼던 그 옵션과 같은 것이다)

**② `bnb_4bit_quant_type='nf4'` — "압축 방식은 NF4로"**

같은 4비트여도 **단계를 어디에 배치하느냐**로 품질이 달라진다.
`'nf4'`는 2-4절에서 검증한 **분포 기반 배치**, `'fp4'`는 단순한 방식이다.
👉 **특별한 이유가 없으면 `'nf4'`** 를 쓴다.

**③ `bnb_4bit_compute_dtype=torch.float16` — "계산할 땐 풀어서"**

저장은 4비트로 하지만, **곱셈·덧셈까지 4비트로 하면 결과가 엉망**이 된다.
그래서 연산 직전에 FP16으로 되돌린다.

```
   저장 :  INT4   (작다)
     ↓  풀기(dequantize)
   계산 :  FP16   (정밀하다)
```

**④ `bnb_4bit_use_double_quant=True` — "남는 것도 마저 압축"**

압축하려면 `scale` 같은 **환산표**를 함께 저장해야 하는데, 이것도 양이 만만치 않다.
이 환산표를 **한 번 더 압축**하는 옵션이다. 켜두면 손해 볼 게 없다.

| 파라미터 | 한 줄 요약 |
|---------|------|
| `load_in_4bit` | 양자화 켜기 |
| `bnb_4bit_quant_type` | 압축 방식 (`'nf4'` 권장) |
| `bnb_4bit_compute_dtype` | 계산할 때 되돌릴 정밀도 (FP16 권장) |
| `bnb_4bit_use_double_quant` | 환산표까지 압축 (켜기 권장) |

<br>

> **💡 `bnb_4bit_compute_dtype`이 중요한 이유**
>
> 가중치는 INT4로 **저장**되지만, 행렬 곱셈 등 실제 **연산**은 FP16으로 수행된다.
> 즉, 추론 시 INT4 → FP16으로 역양자화한 뒤 연산하는 것이다.
> **저장은 작게, 연산은 정밀하게** — 이것이 bitsandbytes의 핵심 전략이다.
>
> ```
>    디스크/GPU 저장 :  INT4  (작다)
>          ↓ dequantize (연산 직전)
>    실제 행렬 곱셈  :  FP16  (정밀하다)
> ```
>
> ⚠️ 이 **dequantize 비용**이 곧 "양자화해도 항상 빨라지지는 않는" 이유다.
> 챕터 4 마지막에서 추론 시간을 직접 재서 확인한다.

> **📌 `bnb_4bit_use_double_quant` — 상수까지 압축한다**
>
> 양자화를 하려면 `scale` 같은 상수를 함께 저장해야 한다.
> 그런데 이 상수도 개수가 많아지면 무시할 수 없는 용량이 된다.
>
> ```
>    가중치를 64개씩 묶어 블록마다 scale 1개 저장
>    -> 파라미터 15억 개면 scale도 약 2,300만 개!
>
>    Double Quantization : 그 scale들을 '다시 한 번' 양자화
>    -> 파라미터당 약 0.4비트 추가 절감
> ```
>
> 작아 보이지만 70B 모델에서는 **수 GB 차이**가 난다.

### 4-4. 실무에서의 양자화 방식 선택

| 방식 | 특징 | 적합한 상황 |
|------|------|------|
| **bitsandbytes** | 런타임 양자화, HuggingFace 연동 | 빠른 실험, 프로토타이핑 (이 실습) |
| **GPTQ** | 사전 양자화, 가중치 보정 | 배포 최적화 |
| **AWQ** | 활성화 기반, 중요 가중치 보존 | 품질 중시 서비스 |
| **GGUF** | llama.cpp 형식, CPU 지원 | 로컬/엣지 추론 |


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: INT4 양자화 모델 로딩 + FP16과의 메모리 비교
#
# [중요: FP16 모델을 먼저 GPU에서 해제한다]
# GPU 메모리에 FP16 모델이 남아 있는 상태에서 INT4를 로딩하면
# 두 모델이 동시에 GPU에 존재하여 메모리가 합산된다.
# 정확한 INT4 단독 메모리를 측정하려면 FP16을 먼저 해제해야 한다.
#
# [BitsAndBytesConfig 파라미터 복습]
# - load_in_4bit: INT4 양자화 활성화
# - bnb_4bit_compute_dtype: 연산 시 사용하는 dtype (FP16 권장)
#   → 가중치는 INT4로 "저장"하되, 행렬곱 등 연산은 FP16으로 수행
# - bnb_4bit_quant_type: 'nf4' = 정규분포 최적화 양자화
# - bnb_4bit_use_double_quant: 양자화 상수를 한 번 더 양자화 (추가 절감)
# ═══════════════════════════════════════════════════════════

# ========== FP16 모델 결과 저장 후 GPU에서 해제 ==========
# 비교를 위해 FP16 결과를 딕셔너리에 저장해 둔다.
fp16_results = {
    'model_memory': model_memory_fp16,
    'peak_memory': peak_fp16,
    'latency': latency_fp16,
    'response': response_fp16,
}

# ★ 핵심: GPU 메모리에서 FP16 모델을 완전히 해제
del model_fp16
torch.cuda.empty_cache()  # GPU 캐시까지 비워야 메모리가 실제로 반환됨
import gc; gc.collect()    # Python 가비지 컬렉터도 실행

print(f'FP16 모델 해제 후 GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

# ========== INT4 양자화 모델 로딩 ==========
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # INT4 양자화
    bnb_4bit_compute_dtype=torch.float16,       # 연산은 FP16
    bnb_4bit_quant_type='nf4',                  # NF4 양자화
    bnb_4bit_use_double_quant=True,             # 이중 양자화
)

print('\nINT4 양자화 모델 로딩 중...')
torch.cuda.reset_peak_memory_stats()

# ★ 핵심: quantization_config를 from_pretrained에 전달
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map='auto',
)

# INT4 모델 단독 메모리 (FP16이 해제된 상태이므로 정확한 값)
model_memory_int4 = torch.cuda.memory_allocated() / 1024**3
print(f'\nINT4 모델 메모리: {model_memory_int4:.2f} GB')
print(f'FP16 모델 메모리: {fp16_results["model_memory"]:.2f} GB (저장된 값)')
print(f'메모리 절감: {(1 - model_memory_int4/fp16_results["model_memory"])*100:.1f}%')

# [예상 결과]
# FP16: ~3.9GB → INT4: ~1.7GB → 약 56% 절감
# 이론적으로는 75% 절감이지만, 양자화 상수(scale, zero_point) 저장과
# 일부 레이어(임베딩 등)가 FP16으로 유지되어 실제 절감률은 이론보다 낮다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: FP16 vs INT4 전체 성능 비교
#
# [비교 지표 3가지]
# 1. 모델 메모리: 가중치만의 크기 → INT4가 확실히 작다
# 2. 피크 메모리: 추론 중 최대값 → 활성화값 때문에 차이가 줄어든다
# 3. 추론 시간: 양자화가 항상 빨라지는 것은 아니다
#    (역양자화 오버헤드가 있어 1.5B 소형 모델에서는 비슷하거나 느릴 수 있음)
# ═══════════════════════════════════════════════════════════

# INT4 추론 실행 (같은 질문으로 비교)
response_int4, latency_int4, peak_int4 = measure_inference(
    model_int4, tokenizer, test_prompt
)

# 저장된 FP16 결과와 비교
fp16_mm = fp16_results['model_memory']
fp16_pk = fp16_results['peak_memory']
fp16_lat = fp16_results['latency']

print('FP16 vs INT4 비교')
print('=' * 70)
print(f'{"지표":<20} {"FP16":>15} {"INT4":>15} {"변화":>15}')
print('-' * 70)
print(f'{"모델 메모리":<20} {fp16_mm:>14.2f}GB {model_memory_int4:>14.2f}GB {(1-model_memory_int4/fp16_mm)*100:>13.1f}%↓')
print(f'{"피크 메모리":<20} {fp16_pk:>14.2f}GB {peak_int4:>14.2f}GB {(1-peak_int4/fp16_pk)*100:>13.1f}%↓')
print(f'{"추론 시간":<20} {fp16_lat:>14.2f}초 {latency_int4:>14.2f}초')
print('=' * 70)

print(f'\nFP16 응답: {fp16_results["response"][:200]}')
print(f'\nINT4 응답: {response_int4[:200]}')

# [결과 해석]
# 모델 메모리: INT4가 확실히 작다 (약 50~60% 절감)
# 피크 메모리: 절감률이 모델 메모리보다 낮다
#   → 이유: 양자화는 가중치만 압축. 활성화값/KV캐시는 FP16 그대로
# 추론 시간: 1.5B 소형 모델에서는 INT4가 반드시 빠르지 않을 수 있다
#   → 이유: INT4→FP16 역양자화 오버헤드가 있기 때문
#   → 7B+ 대형 모델에서는 메모리 절감 효과가 속도 향상으로 이어진다


### 4-5. 모델 메모리 vs 피크 메모리 — 왜 다른가?

비교 결과에서 **모델 메모리 절감은 크지만, 피크 메모리 절감은 작을 수 있다**.
이 차이를 이해하는 것이 양자화 효과를 올바르게 평가하는 데 중요하다.

| 지표 | 측정 시점 | 포함 내용 |
|------|---------|----------|
| **모델 메모리** | 로딩 직후 | 가중치(weights, biases)만 |
| **피크 메모리** | 추론 중 최대 | 가중치 + **활성화값 + KV 캐시 + 임시 버퍼** |

**핵심**: 양자화는 **가중치만 압축**한다. 추론 중 생성되는 활성화값(activations)과
KV 캐시(attention의 Key-Value 저장소)는 여전히 FP16으로 유지된다.
따라서 피크 메모리 절감 효과는 모델 메모리 절감보다 항상 작다.

```
FP16:  모델 ~3.9GB → 피크 ~4.0GB  (차이 ~0.1GB: 활성화 비중 작음)
INT4:  모델 ~1.7GB → 피크 ~3.8GB  (차이 ~2.1GB: 활성화 비중 큼!)
```

> **💡 모델이 클수록 양자화 효과가 극적이다**
>
> 1.5B 모델은 전체 메모리에서 가중치 비중이 상대적으로 작아 효과가 제한적이다.
> 하지만 7B, 13B, 70B 모델에서는 가중치가 전체 메모리의 대부분을 차지하므로
> INT4 양자화 시 **피크 메모리도 극적으로 줄어든다**.
> 실무에서 양자화의 진가는 **큰 모델에서** 드러난다.


---

## 5. 양자화의 부작용 — 환경 변화 입력에서의 품질 저하

![image_D](https://i.ibb.co/WWGsgycX/image-D.png)

### 5-1. 환경 변화(Distribution Shift)란?

LLM은 방대한 텍스트 데이터로 학습된다. 학습 데이터에는 문법적으로 정확하고
명확한 문장이 대부분이다. 하지만 **실제 서비스에서 사용자 입력은 학습 데이터와 다를 수 있다.**

이처럼 학습 데이터와 실제 입력의 **분포가 달라지는 현상**을 `환경 변화(Distribution Shift)`라 한다.

| 유형 | 예시 | 특징 |
|------|------|------|
| **정상** | "서울에서 부산까지 KTX로 얼마나 걸리나요?" | 문법 정확, 맥락 명확 |
| **오타** | "서울에**셔** 부산까지 KTX로 얼마나 걸리나요?" | 학습 데이터에 없는 오타 포함 |
| **노이즈** | "ktx 서울 부산 몇시간? 걍 대충 알려줘 ㅋㅋ" | 불필요한 표현, 구어체 |
| **모호함** | "그거 얼마나 걸려?" | 주어/목적어 생략, 맥락 부족 |
| **조건 추론** | "오후 3시에 서울역에서 KTX 타면 부산에 몇 시에 도착해?" | 복잡한 조건 추가 |

### 5-2. 왜 양자화 모델이 환경 변화에 더 취약한가?

> **💡 먼저 큰 그림 — "작은 오차가 왜 큰 문제가 되는가"**
>
> 2장에서 INT4의 평균 오차가 **0.04 정도**라고 확인했다. 작아 보인다.
> 그런데 왜 답변 품질이 흔들릴까? **세 단계를 거치며 커지기 때문**이다.
>
> ```
>    ① 작은 오차가        15억 개 가중치에      쌓인다        (누적)
>              ↓
>    ② 그 결과가          Attention 단계에서    증폭된다      (확대)
>              ↓
>    ③ 원래 아슬아슬하던  '드문 입력'에서       티가 난다     (표출)
> ```
>
> 👉 **평소엔 괜찮다가, 어려운 입력에서만 무너지는 이유**가 여기 있다.
> 아래에서 세 단계를 하나씩 본다.

FP16 모델도 환경 변화에 완벽하지 않지만, **양자화 모델은 더 심하게** 품질이 떨어진다.

#### ① 정밀도 손실 누적

양자화는 각 가중치에서 **작은 오차**를 발생시킨다.
이 오차가 수십억 개의 파라미터 전체에 걸쳐 누적되면 출력에 영향을 미친다.

```
FP16 가중치:  [0.123456, 0.234567, 0.345678, ...]
                  ↓ 양자화
INT4 가중치:  [0.125000, 0.234375, 0.343750, ...]  ← 각각 작은 오차
                  ↓ 수십억 번 연산
최종 출력: 오차가 누적되어 다른 결과 가능
```

#### ② Attention Score 변화

LLM의 핵심인 Self-Attention은 토큰 간 관계를 계산한다.
양자화 오차로 attention score가 미세하게 달라지면, **다른 토큰에 집중**하게 된다.

> **💡 2-1 챕터에서 배운 것이 여기서 쓰인다**
>
> Attention은 점수를 **softmax**로 확률로 바꾼다.
> softmax는 **입력 차이를 지수적으로 증폭**하는 함수다.
>
> ```
>    점수 차이 0.1  ->  softmax 후 확률 차이는 그보다 크게 벌어진다
> ```
>
> 즉 **가중치의 작은 오차가 attention 단계에서 확대**될 수 있다.
> 이것이 "작은 오차인데 왜 출력이 달라지는가"에 대한 답이다.

```
입력: "서울에셔 부산까지" (오타 포함)

FP16:  "에셔"를 "에서"와 유사하게 인식 (높은 attention)
INT4:  미세한 차이로 "에셔"를 다르게 인식 (낮은 attention) → 잘못된 해석
```

#### ③ 경계 케이스(Edge Case) 처리 능력 저하

> **💡 비유: 실력에 여유가 있느냐**
>
> ```
>    FP16  :  시험을 90점으로 통과하던 학생   -> 문제가 조금 어려워져도 80점, 통과
>    INT4  :  간신히 61점으로 통과하던 학생   -> 조금만 어려워지면 59점, 탈락
> ```
>
> **평소 문제(자주 본 패턴)** 는 둘 다 잘 푼다.
> 차이는 **어려운 문제(드문 패턴)** 에서 드러난다.

| 입력 패턴 | FP16 | INT4 |
|---------|:---:|:---:|
| 자주 본 패턴 ("서울에서 부산까지") | ✅ 강함 | ✅ 강함 |
| 드문 패턴 (오타, 모호함) | ✅ 적당히 처리 | ❌ 취약 |

자주 본 패턴은 양자화 후에도 강하게 유지되지만,
`드물게 본 패턴(경계 케이스)`은 정밀도 손실의 영향을 더 크게 받는다.

> 👉 이 **"여유가 줄어든 상태"** 를 챕터 6에서는 **추론 여유(inference headroom)** 라고 부른다.
> 그리고 **프롬프트로 그 여유를 보충해 주는 것**이 챕터 6의 전략이다.

아래에서 이 현상을 직접 확인해 보자.

### 5-3. 실습에서 무엇을 관찰해야 하는가?

양자화 모델의 품질 저하는 `"답변을 못 하는 것"`이 아니라 `"미묘하게 이상한 답변을 하는 것"`이다.
최신 모델(Qwen2.5 등)은 단순 오타 정도는 잘 처리하므로, 다음과 같은 **미묘한 이상 징후**를 관찰해야 한다.

| 이상 징후 | 설명 | 서비스 영향 |
|---------|------|------|
| **외국어 혼입** | 한국어 답변에 중국어/일본어 문자 등장 (예: 高速鉄路) | 사용자 혼란, 신뢰도 하락 |
| **없는 정보 생성** | 존재하지 않는 노선, 서비스, 요금을 지어냄 | 잘못된 안내, 클레임 |
| **맥락 이탈** | 질문 주제를 벗어난 답변 (KTX → 항공편) | 답변 품질 저하 |
| **답변 불안정** | 중간 끊김, 반복, 비일관적 형식 | 전문성 의심 |

<br>

> **💡 "정답을 맞히는 것"과 "서비스 품질"은 다르다**
>
> "약 2시간 30분"이라는 핵심 정보를 맞히더라도, 답변에 중국어가 섞이거나
> 없는 서비스를 언급하면 고객 서비스로서는 실격이다.
> 챕터 6의 프롬프트 강화는 이런 **미묘한 품질 문제**를 보완하는 전략이다.

> **⚠️ 결과가 매번 다를 수 있다 — 그래서 '측정'이 필요하다**
>
> 이 실험은 **한 번 돌려서 판단하기 어렵다.**
> 모델·라이브러리 버전, 질문에 따라 품질 저하가 뚜렷할 수도, 거의 안 보일 수도 있다.
>
> 5-1에서 배운 교훈을 떠올리자.
>
> > **"1건만 보고 판단하지 마라. 몇 건으로 측정했는지 물어라."**
>
> 👉 그래서 아래 실습에는 **품질 이상을 자동으로 집계하는 코드**를 함께 두었다.
> 눈으로 훑는 대신 **숫자로 확인**하자.


### 5-4. ⭐ 눈으로 훑지 말고 숫자로 재자 — 품질 자동 판정

"중국어가 섞였다", "답변이 이상하다"를 **사람이 하나하나 읽어서 판단**하면
주관적이고, 건수가 늘어나면 감당할 수 없다.

**기계적으로 잡아낼 수 있는 이상 징후는 코드로 판정**하는 것이 좋다.

| 징후 | 탐지 방법 | 한계 |
|---|---|---|
| **CJK 혼입** | 한자·가나 유니코드 범위 검사 | 정확 ✅ |
| **반복** | 같은 문자열이 다시 등장하는지 | 대체로 정확 |
| **중간 끊김** | 문장 종결부호로 안 끝남 | 토큰 한도 때문일 수도 |
| 없는 정보 생성 | — | ❌ **자동 판정 불가** (사람이 봐야 함) |

> **⚠️ 자동 판정은 만능이 아니다**
>
> "존재하지 않는 노선을 지어냈다" 같은 **사실 오류**는 코드로 못 잡는다.
> 그건 4-1에서 배운 **LLM as Judge**나 사람의 검수가 필요하다.
>
> 👉 **자동 판정은 1차 필터**다. 이것도 4-1·4-2에서 반복해서 나온 원칙이다.
> **자동화는 사람을 없애는 게 아니라, 사람이 봐야 할 양을 줄인다.**


In [ ]:
import re

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 응답 품질 이상을 '자동으로' 판정하는 함수
#
# [왜 필요한가]
# 양자화의 품질 저하는 "답을 못 하는 것"이 아니라 "미묘하게 이상한 것"이다.
# 눈으로 훑으면 놓치기 쉽고, 사람마다 판단이 다르다.
# 기계적으로 잡을 수 있는 것은 코드로 판정해 '숫자'로 만든다.
#
# [탐지 항목]
#   ① CJK 혼입  : 한국어 답변에 한자/일본어 가나가 섞였는가
#   ② 반복      : 같은 문구가 되풀이되는가
#   ③ 중간 끊김  : 문장이 종결되지 않고 잘렸는가
#
# ⚠️ "없는 정보를 지어냈는가"는 코드로 판정할 수 없다. 사람이 봐야 한다.
# ═══════════════════════════════════════════════════════════

# 한자(4E00-9FFF) + 일본어 가나(3040-30FF)
# 한글(AC00-D7A3)은 제외 -> 한국어 답변은 정상으로 판정된다
CJK_PATTERN = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff]')


def check_quality(text, repeat_len=12):
    """응답 텍스트에서 기계적으로 탐지 가능한 이상 징후를 찾는다.

    Returns: dict — 항목별 판정 결과
    """
    # ① CJK 혼입: 한자/가나가 하나라도 있으면 이상
    cjk_chars = CJK_PATTERN.findall(text)

    # ② 반복: 길이 repeat_len짜리 조각이 뒤에서 또 나오는지
    has_repeat = False
    for i in range(max(0, len(text) - repeat_len * 2)):
        chunk = text[i:i + repeat_len]
        if chunk.strip() and chunk in text[i + repeat_len:]:
            has_repeat = True
            break

    # ③ 중간 끊김: 문장 종결부호로 끝나지 않음
    stripped = text.strip()
    truncated = bool(stripped) and stripped[-1] not in '.。!?！？…"\')'

    return {
        'cjk': len(cjk_chars),          # 혼입된 CJK 글자 수
        'cjk_chars': ''.join(sorted(set(cjk_chars)))[:20],
        'repeat': has_repeat,
        'truncated': truncated,
        'length': len(stripped),
    }


def print_quality(text, label=''):
    """판정 결과를 한 줄로 출력한다."""
    q = check_quality(text)
    flags = []
    if q['cjk']:
        flags.append(f'CJK혼입 {q["cjk"]}자({q["cjk_chars"]})')
    if q['repeat']:
        flags.append('반복')
    if q['truncated']:
        flags.append('끊김')
    status = '❌ ' + ' / '.join(flags) if flags else '✅ 이상 없음'
    print(f'  판정{label}: {status}')
    return q


# ========== 동작 확인 ==========
print('품질 판정 함수 테스트')
print('=' * 60)
samples = [
    ('정상 한국어', '서울에서 부산까지 KTX로 약 2시간 30분 걸립니다.'),
    ('중국어 혼입', '서울에서 부산까지 高速鉄路로 약 2시간 걸립니다.'),
    ('영어 섞임',   'KTX로 약 2시간 30분(2h 30m) 소요됩니다.'),
    ('문장 끊김',   '서울에서 부산까지 KTX로 약 2시간 30분 정도 소요되며 요금은'),
]
for name, text in samples:
    print(f'\n[{name}] {text}')
    print_quality(text)

print('\n' + '=' * 60)
print('→ 영어가 섞인 것은 정상으로 판정된다. 한자/가나만 문제 삼는다.')
print('  이제 이 함수로 챕터 5·6의 응답을 자동 채점한다.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 환경 변화 입력에서 INT4 모델의 품질 저하 관찰
#
# [환경 변화(Distribution Shift)란?]
# 학습 데이터에는 문법적으로 정확한 문장이 대부분이다.
# 하지만 실제 사용자 입력에는 오타, 구어체, 맥락 부족 등이 포함된다.
# 이런 입력이 학습 데이터의 "분포"와 다르므로 "환경 변화"라 부른다.
#
# [관찰 가이드 — 무엇을 봐야 하는가?]
# 양자화 모델은 "답변을 아예 못 하는" 것이 아니라
# "미묘하게 이상한 답변"을 하는 경우가 많다.
# 아래 체크리스트를 기준으로 관찰하라:
#   □ 한국어 질문에 중국어/일본어 문자가 섞여 나오는가? (예: 高速鉄路)
#   □ 존재하지 않는 정보를 지어내는가? (예: KTX E, 350km/h 등)
#   □ 질문의 맥락을 벗어나는가? (예: KTX를 물었는데 항공사를 답변)
#   □ 답변이 중간에 끊기거나 반복되는가?
#   □ "정확한 듯 보이지만 틀린 정보"가 포함되어 있는가?
#
# 이런 미묘한 품질 저하가 실제 서비스에서는 심각한 문제가 된다.
# 고객이 받는 답변에 중국어가 섞이거나, 없는 서비스를 안내하면
# 신뢰도가 급격히 하락하기 때문이다.
# ═══════════════════════════════════════════════════════════

test_samples = [
    # 정상: 기준선 (이 결과와 나머지를 비교)
    {'type': '정상',
     'prompt': '서울에서 부산까지 KTX로 얼마나 걸리나요?'},

    # 오타: 여러 글자에 오타를 섞어 난이도를 높임
    {'type': '오타',
     'prompt': '서울에셔 부산까지 KTX로 얼마나 걸리나요?'},

    # 구어체 + 줄임말: 실제 채팅에서 흔한 형태
    {'type': '구어체',
     'prompt': 'ktx 서울 부산 몇시간? 걍 대충 알려줘 ㅋㅋ'},

    # 맥락 부족 + 복합 질문: 양자화 모델이 추론 여유 부족으로 어려워하는 유형
    {'type': '복합질문',
     'prompt': '서울에서 부산 가는데 KTX랑 SRT 중에 뭐가 더 빠르고 싼지 비교해줘'},

    # 조건 질문 (산술 없음): 상황 판단이 필요한 질문
    {'type': '조건판단',
     'prompt': '주말에 서울역에서 KTX 자유석 타면 앉을 수 있을까요?'},

    # 숫자/조건 포함: 정확한 산술 추론이 필요한 질문
    # ※ 양자화 모델은 시간 계산 같은 산술 추론에서 특히 오류가 크다.
    #    이 유형은 TTP로도 교정이 어렵다 (TTP는 방향 제시이지, 계산 교정이 아님)
    {'type': '산술추론',
     'prompt': '오후 3시에 서울역에서 KTX 타면 부산에 몇 시에 도착해?'},
]

print('INT4 모델 — 환경 변화 입력 테스트')
print('=' * 70)

env_results = []
for sample in test_samples:
    response, latency, _ = measure_inference(
        model_int4, tokenizer, sample['prompt'], max_new_tokens=150
    )
    env_results.append({'type': sample['type'], 'response': response})
    print(f'\n[{sample["type"]}]')
    print(f'입력: {sample["prompt"]}')
    print(f'응답: {response[:300]}')
    print_quality(response)      # ★ 자동 품질 판정 (앞 셀에서 정의한 함수)

# ========== 자동 판정 집계 ==========
# ★ 개별 결과를 눈으로 훑는 대신, 전체를 숫자로 요약한다.
n_cjk = sum(1 for r in env_results if check_quality(r['response'])['cjk'] > 0)
n_rep = sum(1 for r in env_results if check_quality(r['response'])['repeat'])
n_tot = len(env_results)

print('\n' + '=' * 70)
print(f'자동 판정 요약 (총 {n_tot}건)')
print(f'  CJK 혼입 : {n_cjk}/{n_tot}건')
print(f'  반복 발생 : {n_rep}/{n_tot}건')
print('  ⚠️ "없는 정보 생성"은 자동 판정 불가 — 아래 체크리스트로 직접 확인할 것')

print('\n' + '=' * 70)
print()
print('📋 관찰 체크리스트 — 아래 항목이 발견되는지 확인하라:')
print('  □ 한국어 답변에 중국어/일본어 문자가 섞여 있는가? (예: 高速鉄路, 铁道)')
print('  □ 존재하지 않는 정보를 생성했는가? (예: 없는 노선, 틀린 요금)')
print('  □ 질문 맥락을 벗어난 답변이 있는가? (예: KTX를 물었는데 비행기 설명)')
print('  □ 답변이 중간에 끊기거나 같은 말을 반복하는가?')
print('  □ 구어체/줄임말을 이해하지 못하고 엉뚱한 해석을 했는가?')
print()
print('⚠️ 참고: 최신 모델(Qwen2.5)은 단순 오타 정도는 잘 처리할 수 있다.')
print('  하지만 "미묘하게 이상한 답변" (중국어 혼입, 없는 정보 생성 등)은')
print('  실제 고객 서비스에서 신뢰도를 크게 떨어뜨리는 문제이다.')
print('  TTP는 이런 미묘한 품질 저하를 보완하는 전략이다.')


---

## 6. TTP(Test-time Prompting)로 품질 복구

![image_E](https://i.ibb.co/xnVVwVF/image-E.png)

### 6-1. 추론 시점의 프롬프트 강화 (TTP)

**Test-time Prompting(TTP)** = 추론 시점에 **프롬프트를 강화**하여 품질을 안정화하는 전략

핵심: **모델을 재학습하지 않고**, 프롬프트 엔지니어링만으로 양자화 품질 저하를 보완한다.

| 전략 | 방법 | 효과 |
|------|------|------|
| **TTP-A** | 출력 규칙/제약을 명시 | 불명확한 입력에서도 방향성 제시 |
| **TTP-B** | few-shot 예시 포함 | 원하는 출력 형식을 "시연"으로 학습 |

> **📌 용어에 대하여 — 새로운 기술이 아니다**
>
> "TTP"는 이 실습에서 **양자화 품질 보완이라는 목적**에 초점을 맞춰 부르는 이름이다.
> 기법 자체는 **2-2 챕터에서 이미 배운 프롬프팅 기법**과 같다.
>
> | 이번 챕터 | 2-2에서 배운 이름 |
> |---|---|
> | **TTP-A** (규칙 명시) | **Role Prompting** + 제약 조건 명시 |
> | **TTP-B** (예시 제공) | **Few-shot** |
>
> ⭐ **새 기술을 배우는 게 아니라, 이미 아는 기법을 새 문제에 적용하는 것**이다.
> 그리고 이 관점이 중요하다 — **양자화로 생긴 문제를 프롬프트로 푼다.**
> 모델을 손대지 않고 입력만 바꿔서 해결하는 것이 이 전략의 가치다.

### 6-2. TTP가 효과적인 원리

양자화된 모델은 정밀도 손실로 **"추론 여유(inference headroom)"가 줄어든 상태**이다.

"추론 여유"란 모델이 불명확한 입력을 받았을 때 여러 해석 중에서
올바른 것을 골라내는 능력이다. FP16은 이 여유가 충분하지만,
INT4는 정밀도 손실로 인해 여유가 줄어 잘못된 방향을 선택할 확률이 높아진다.

```
불명확한 입력: "그거 얼마나 걸려?"

FP16: 여러 해석 후보 중 맥락에 맞는 것 선택 (추론 여유 ☀️)
INT4: 잘못된 해석 선택 가능 (추론 여유 부족 🌧️)

TTP 적용: "교통 관련 질문입니다. 소요 시간을 답하세요."
INT4 + TTP: 명확한 방향이 제시되어 올바른 추론 유도 (☀️ 복구)
```

### 6-3. TTP 전략별 동작 원리

**TTP-A (규칙 명시)**

"오타가 있어도 의도를 파악하세요", "간결하게 핵심만 답하세요" 같은
**명시적 규칙**을 프롬프트에 포함시킨다.
모델이 어떤 방향으로 추론해야 하는지 **나침반**을 제공하는 효과가 있다.

**TTP-B (Few-shot 예시)**

"Q: 서울→대전 KTX? A: 약 50분~1시간" 같은 **예시 1~2개**를 프롬프트에 포함시킨다.
모델은 예시를 보고 "이런 형식으로 답하면 되는구나"를 즉석에서 학습한다.
출력 형식의 일관성을 높이는 데 특히 효과적이다.

### 6-4. TTP-A와 TTP-B는 역할이 다르다

두 전략은 **보완하는 영역이 다르다.** 실습 결과를 관찰할 때 이 차이를 인식해야 한다.

| 전략 | 핵심 지시 | 오타 교정 | 출력 형식 통일 | 비유 |
|------|---------|:---:|:---:|------|
| **TTP-A** | "오타가 있어도 **의도를 파악**하세요" | ✅ 강함 | △ 약함 | **나침반** (방향 제시) |
| **TTP-B** | "이 **예시와 동일한 형식**으로 답하세요" | △ 약함 | ✅ 강함 | **샘플 답안** (형식 제시) |

예를 들어, 입력에 오타("서울에셔")가 포함된 경우:
- **TTP-A**: "오타가 있어도 의도를 파악하세요"라는 규칙이 있으므로 → "서울에서"로 교정하여 답변
- **TTP-B**: 오타 교정 규칙이 없으므로 → "서울에셔"를 그대로 가져다 쓸 수 있음 (출력 형식은 통일됨)

> **💡 실무에서는 TTP-A + TTP-B를 결합하여 사용한다**
>
> 규칙 명시(A)와 few-shot 예시(B)를 하나의 템플릿에 함께 포함시키면
> **오타 교정 + 출력 형식 통일**을 동시에 달성할 수 있다.
>
> 2-2에서 배운 **"기법은 조합해서 쓴다"** 는 원칙 그대로다.

> **⚠️ 프롬프트로 풀 수 없는 것도 있다 — 산술 추론**
>
> "오후 3시에 출발하면 몇 시 도착?" 같은 **계산**은 프롬프트를 아무리 강화해도
> 정확해지지 않는다. TTP는 **"어느 방향으로 추론할지"** 를 알려줄 뿐,
> **계산 능력 자체를 보완하지는 못하기** 때문이다.
>
> ```
>    프롬프트로 해결 가능  :  형식 통일, 의도 파악, 맥락 보완
>    프롬프트로 해결 불가  :  정확한 계산, 최신 사실, 외부 데이터
> ```
>
> 👉 **그럼 계산이 중요한 서비스는?**
> **4-2에서 배운 Tool-use**로 계산기를 붙이는 것이 정답이다.
> 아래 실습에서 이 한계를 **직접 관찰**하게 된다.

### 6-5. 효과적인 TTP 템플릿 설계 원칙

| 원칙 | Bad 예시 | Good 예시 |
|------|---------|----------|
| **규칙 명시** | "질문에 답해주세요" | "오타가 있어도 의도를 파악하세요. 간결하게 핵심만 답하세요." |
| **출력 형식 지정** | "시간을 알려주세요" | "소요 시간을 '약 X시간 Y분' 형식으로 답하세요" |
| **Few-shot 예시** | (없음) | "예시: Q: 서울→대전 KTX? A: 약 50분~1시간" |

<br>

> **💡 TTP의 한계**
>
> TTP는 프롬프트 길이가 늘어나므로 **입력 토큰 수가 증가**한다.
> 이는 추론 비용 증가로 이어질 수 있다.
> 따라서 "메모리 절감 + 약간의 토큰 증가"와 "품질 유지" 사이의
> 트레이드오프를 고려하여 적용해야 한다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: TTP(Test-time Prompting) 템플릿 + 효과 비교
#
# [TTP = 프롬프트 강화로 양자화 품질 저하를 보완하는 전략]
# 모델을 재학습하지 않고, 입력 프롬프트에 "규칙"이나 "예시"를 추가하여
# 양자화 모델의 "추론 여유" 부족을 보완한다.
#
# [TTP-A: 규칙 명시]
# "오타가 있어도 의도를 파악하세요" 같은 명시적 규칙을 포함.
# 모델에게 "이 방향으로 추론하라"는 나침반을 제공하는 효과.
#
# [TTP-B: Few-shot 예시]
# "Q: 서울→대전? A: 약 50분" 같은 예시를 1~2개 포함.
# 모델이 예시를 보고 "이런 형식으로 답하면 되는구나"를 즉석 학습.
# 출력 형식의 일관성을 높이는 데 특히 효과적.
#
# [관찰 포인트]
# - TTP 없음: 오타/모호함에서 품질 저하 발생
# - TTP-A: 규칙 명시로 방향성이 잡히는지 확인
# - TTP-B: 예시로 출력 형식이 통일되는지 확인
# ═══════════════════════════════════════════════════════════

# ========== TTP 템플릿 정의 ==========

# TTP-A: 출력 규칙/제약 강화
# → {question}에 사용자 질문이 삽입된다 (str.format() 사용)
TTP_A = '''다음 질문에 답해주세요.

**규칙:**
1. 오타나 불명확한 표현이 있어도 의도를 파악하세요.
2. 교통 관련 질문은 대략적인 소요 시간(시간 단위)으로 답하세요.
3. 맥락이 부족하면 일반적인 상황을 가정하여 답하세요.
4. 간결하게 핵심만 답하세요.

**질문:** {question}

**답변:**'''

# TTP-B: few-shot 예시 포함
# → 예시 1개만으로도 출력 형식 통일에 효과적
TTP_B = '''다음은 교통 관련 질문과 답변의 예시입니다.

**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분~1시간 정도 소요됩니다.

이제 아래 질문에 동일한 형식으로 답해주세요.

**질문:** {question}

**답변:**'''

# ========== 환경 변화 입력에 TTP 적용 ==========
ttp_test = [
    {'type': '오타', 'prompt': '서울에셔 부산까지 KTX로 얼마나 걸리나요?'},
    {'type': '구어체', 'prompt': 'ktx 서울 부산 몇시간? 걍 대충 알려줘 ㅋㅋ'},
    # 조건 판단: 산술 없이 상황 판단이 필요한 질문 → TTP 효과가 잘 드러남
    {'type': '조건판단', 'prompt': '주말에 서울역에서 KTX 자유석 타면 앉을 수 있을까요?'},
    # 산술 추론: 시간 계산이 필요한 질문
    # ※ TTP로도 산술 오류는 교정이 어렵다.
    #    TTP는 "추론 방향 제시"이지 "계산 능력 보완"이 아니기 때문이다.
    #    이 결과를 통해 TTP의 한계를 함께 관찰할 수 있다.
    {'type': '산술추론', 'prompt': '오후 3시에 서울역에서 KTX 타면 부산에 몇 시에 도착해?'},
]

print('TTP 적용 효과 비교')
print('=' * 70)

ttp_results = []   # 자동 판정 집계용

for sample in ttp_test:
    print(f'\n[{sample["type"]}] 입력: {sample["prompt"]}')
    print('-' * 50)

    # TTP 없음: 원본 프롬프트 그대로
    r0, _, _ = measure_inference(model_int4, tokenizer, sample['prompt'], max_new_tokens=80)
    print(f'TTP 없음: {r0[:150]}')
    print_quality(r0, ' ')      # ★ 자동 품질 판정

    # TTP-A: 규칙 명시 → format()으로 {question}에 질문을 삽입
    r1, _, _ = measure_inference(model_int4, tokenizer, TTP_A.format(question=sample['prompt']), max_new_tokens=80)
    print(f'TTP-A:    {r1[:150]}')
    print_quality(r1, ' ')

    # TTP-B: few-shot 예시 포함
    r2, _, _ = measure_inference(model_int4, tokenizer, TTP_B.format(question=sample['prompt']), max_new_tokens=80)
    print(f'TTP-B:    {r2[:150]}')
    print_quality(r2, ' ')

    # 이번 샘플의 결과를 집계용으로 저장
    ttp_results.append({'type': sample['type'], 'none': r0, 'a': r1, 'b': r2})

# ========== 자동 판정 집계 ==========
# ★ 개별 응답을 눈으로 훑는 대신, 세 조건을 '숫자'로 비교한다.
#   5-1에서 배운 원칙 — "1건만 보고 판단하지 마라"
n = len(ttp_results)
print('\n' + '=' * 70)
print(f'자동 판정 집계 (총 {n}건)')
print(f'{"조건":<12}{"CJK 혼입":>12}{"반복":>10}{"끊김":>10}')
print('-' * 46)
for key, label in [('none', 'TTP 없음'), ('a', 'TTP-A'), ('b', 'TTP-B')]:
    qs = [check_quality(r[key]) for r in ttp_results]
    print(f'{label:<12}{sum(1 for q in qs if q["cjk"]>0):>10}건'
          f'{sum(1 for q in qs if q["repeat"]):>8}건'
          f'{sum(1 for q in qs if q["truncated"]):>8}건')
print('-' * 46)
print('→ 숫자가 줄었다면 프롬프트 강화가 실제로 효과를 냈다는 뜻이다.')
print('  ⚠️ 4건은 통계적으로 적다. 판단하려면 더 많은 샘플이 필요하다. (5-1의 교훈)')

# [결과 해석]
# TTP 없음: 오타에서 엉뚱한 답, 모호함에서 맥락 추론 실패
# TTP-A: 규칙이 명시되어 "교통 질문"이라는 방향성이 잡힘
# TTP-B: 예시가 있어 출력 형식까지 통일됨 (가장 안정적)
# → 재학습 없이 프롬프트만으로 품질이 회복되는 것이 핵심 가치
print('\n' + '=' * 70)
print('→ TTP를 적용하면 오타/모호한 입력에서도 응답 품질이 안정화된다.')
print('  TTP-B(few-shot)는 출력 형식까지 통일시켜 가장 일관된 결과를 만든다.')
print('  핵심: 모델 재학습 없이 프롬프트 엔지니어링만으로 품질을 복구할 수 있다.')
print()
print('📋 비교 관찰 포인트:')
print('  □ 중국어/일본어 혼입이 사라졌는가?')
print('  □ 존재하지 않는 정보 생성이 줄었는가?')
print('  □ 답변 형식이 일관되게 통일되었는가? (특히 TTP-B)')
print('  □ 구어체/줄임말도 올바르게 해석하는가?')
print('  □ 조건판단(자유석): TTP 적용 후 답변이 더 구체적인가?')
print()
print('⚠️ TTP의 한계:')
print('  산술추론(몇 시 도착?) 결과를 보면, TTP를 적용해도 시간 계산이 부정확할 수 있다.')
print('  TTP는 "추론 방향 제시"이지 "계산 능력 보완"이 아니기 때문이다.')
print('  산술 정확도가 중요한 서비스에서는 양자화 수준을 보수적으로 선택하거나,')
print('  외부 계산 도구(Tool)를 연결하는 Agent 방식이 필요하다. (4-2에서 배운 내용)')
print()
print('⭐ 오늘의 큰 그림:')
print('  프롬프트로 풀 수 있는 문제  -> 프롬프트로 (5-2 챕터 6)')
print('  프롬프트로 못 푸는 문제     -> 도구를 붙인다 (4-2 Tool-use)')
print('  지식이 부족한 문제          -> 자료를 준다   (4-1 RAG)')
print('  형식·말투가 문제            -> 학습시킨다     (5-1 Fine-tuning)')
print('  => 문제에 맞는 기법을 고르는 것이 실력이다.')


---

## 7. 정리

### 오늘 배운 전체 흐름

```
① LLM은 크다 → GPU 메모리 부족 문제                        (챕터 1)
② 비트 수를 줄여 메모리 절감 — FP32 → FP16 → INT4            (챕터 2)
③ 균등 분할보다 '분포 기반' 배치가 낫다 — NF4의 아이디어 검증   (챕터 2-4)
④ FP16 기준선 측정 — 비교 대상 확보                          (챕터 3)
⑤ INT4 양자화 — bitsandbytes로 메모리 절감 확인               (챕터 4)
⑥ 양자화의 부작용 — 환경 변화 입력에서 품질 저하               (챕터 5)
⑦ 프롬프트 강화로 복구 — 그리고 그 한계                       (챕터 6)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **Quantization** | 가중치의 비트 수를 줄여 메모리 절감 (FP16 → INT4) |
| **양자화 오차** | `round()`로 인한 반올림 오차. 단계가 적을수록 커진다 |
| **NF4** | 정규분포에 맞춰 **대표값을 배치**한 4비트. 같은 비트로 오차가 적다 |
| **PTQ** | 학습 완료 후 양자화. 재학습 불필요. 가장 간편 |
| **Double Quant** | 양자화 상수까지 한 번 더 압축 |
| **환경 변화** | 오타·구어체·모호함 등 학습 분포와 다른 입력 |
| **TTP** | 추론 시 프롬프트 강화로 품질 보완 (= 2-2의 프롬프팅 기법) |

### ⚠️ 이번 챕터에서 꼭 기억할 3가지

| # | 내용 |
|:--:|---|
| 1 | **양자화는 가중치만 압축한다** — 활성화값·KV 캐시는 그대로라 피크 메모리 절감은 작다 |
| 2 | **양자화해도 항상 빨라지지 않는다** — 역양자화 비용 때문. 큰 모델일수록 유리 |
| 3 | **프롬프트로 못 푸는 문제가 있다** — 산술은 Tool을 붙여야 한다 |

---

## 🎓 전체 과정 회고 — 여기까지 온 길

이번이 정규 커리큘럼의 **마지막 챕터**다. 지금까지 배운 것을 한 번에 이어 보자.

```
[1장] 머신러닝 · 딥러닝의 기초
   1-1  데이터를 이해하는 법 (EDA, train/test 분리)
   1-2  신경망은 어떻게 학습하는가 (순전파 → 손실 → 역전파 → 업데이트)
                    ↓
[2장] 언어를 다루는 법
   2-1  토큰화와 임베딩 — 텍스트를 숫자로
   2-2  프롬프팅과 합성 데이터 — LLM을 다루는 법
                    ↓
[3장] 이미지와 전이학습
   3-1  CNN과 전이학습 — "이미 배운 것을 활용한다"
                    ↓
[4장] LLM 애플리케이션
   4-1  RAG — 모델에게 '자료'를 준다
   4-2  Agent — 모델에게 '손'을 달아준다
                    ↓
[5장] 모델을 직접 다루기
   5-1  PEFT/QLoRA — 모델을 '학습'시킨다
   5-2  Quantization — 모델을 '가볍게 서빙'한다   ← 지금 여기
```

### ⭐ 하나의 질문으로 꿰어보기

> **"LLM이 내가 원하는 대로 동작하지 않는다. 어떻게 할 것인가?"**

이 질문에 대한 답이 **챕터마다 다르게** 주어졌다.

| 문제 | 해법 | 챕터 |
|---|---|:---:|
| 프롬프트가 부실하다 | **프롬프팅 기법** (Few-shot, CoT, Role) | 2-2 |
| 우리 회사 정보를 모른다 | **RAG** — 자료를 검색해서 준다 | 4-1 |
| 외부 도구·최신 정보가 필요하다 | **Agent** — 도구를 쥐여준다 | 4-2 |
| 말투·형식이 안 맞는다 | **Fine-tuning** — 학습시킨다 | 5-1 |
| GPU 메모리가 부족하다 | **Quantization** — 압축한다 | 5-2 |

> **⭐ 가장 중요한 것은 "무엇을 쓸지 고르는 판단력"이다**
>
> 오늘 마지막 실습에서 확인했듯,
> **산술 오류는 프롬프트로 못 고치고 Tool이 필요**했다.
>
> ```
>    비싼 기법이 항상 좋은 것이 아니다
>    프롬프팅 -> RAG -> Fine-tuning -> 양자화 배포
>    앞의 것으로 해결되면 뒤로 갈 이유가 없다
> ```
>
> 이 과정에서 여러 번 반복해서 나온 원칙이다.
> **기법을 아는 것보다, 언제 무엇을 쓸지 아는 것이 실력이다.**

### 🔁 반복해서 나왔던 원칙들

| 원칙 | 처음 나온 곳 | 다시 나온 곳 |
|---|---|---|
| **데이터를 먼저 보라 (EDA)** | 1-1 | 2-2, 5-1(과제 필터링), 5-2(NF4) |
| **train/test를 분리하라** | 1-1 | 3-1, 5-1 |
| **측정 없이는 개선도 없다** | 1-1 | 5-1(1건 vs 20건), 5-2(자동 판정) |
| **모르면 모른다고 하라 (환각)** | 2-2 | 4-1, 4-2, 5-1 |
| **자동화는 사람을 대체하지 않는다** | 4-1(LLM as Judge) | 4-2(HITL), 5-2(자동 판정) |
| **싼 방법부터 시도하라** | 3-1(LP → FT) | 5-1, 5-2 |

---

### 🔬 직접 해볼 실험

| # | 실험 | 바꿀 것 | 관찰할 것 |
|:--:|---|---|---|
| 1 | INT8과 비교 | `load_in_8bit=True` | 메모리·품질이 INT4와 어떻게 다른가 |
| 2 | NF4 vs FP4 | `bnb_4bit_quant_type='fp4'` | 분포 최적화가 실제로 유리한가 |
| 3 | Double Quant 끄기 | `bnb_4bit_use_double_quant=False` | 메모리가 얼마나 늘어나는가 |
| 4 | **더 큰 모델** | `Qwen2.5-7B-Instruct` | ⭐ 양자화 효과가 극적으로 커지는가 |
| 5 | **평가 건수 확대** | 테스트 샘플 20건 이상 | ⭐ 품질 저하가 통계적으로 보이는가 |
| 6 | TTP-A + B 결합 | 두 템플릿을 합치기 | 오타 교정 + 형식 통일이 동시에 되는가 |

> **⭐ 4번과 5번을 권장합니다**
>
> **4번**: 1.5B에서는 양자화 효과가 제한적이다. 7B로 올리면
> "양자화의 진가는 큰 모델에서 드러난다"를 직접 확인할 수 있다.
> ⚠️ 단, FP16 7B는 ~14GB라 16GB에서 빠듯하다. INT4만 로딩해 보는 것도 방법이다.
>
> **5번**: 지금은 6건뿐이라 품질 저하를 단정하기 어렵다.
> 5-1에서 배운 그대로 — **표본이 적으면 결론을 낼 수 없다.**

### 🐛 자주 만나는 문제

| 증상 | 원인 | 해결 |
|---|---|---|
| `bitsandbytes` import 실패 | CUDA GPU 없음 | GPU 환경 필요. CPU에서는 동작 불가 |
| INT4 메모리가 예상보다 큼 | FP16이 GPU에 남아 있음 | `del model_fp16` + `empty_cache()` 확인 |
| `CUDA out of memory` | 두 모델 동시 로딩 | 커널 재시작 후 순서대로 실행 |
| INT4가 FP16보다 느림 | **정상** | 역양자화 오버헤드. 소형 모델에서 흔하다 |
| 품질 저하가 안 보임 | 최신 모델이 견고함 | 정상. 자동 판정 숫자로 확인할 것 |
| 절감률이 이론값(75%)보다 낮음 | **정상** | 양자화 상수 + 일부 층은 FP16 유지 |

### 자기주도 실습 안내

이제 `실습_5-2_Quantization.ipynb`를 열고,
오늘 배운 개념을 TODO 코드로 직접 구현해 보자.

---

> ### 🎉 수고하셨습니다
>
> Python 기초에서 시작해 **직접 모델을 학습시키고 배포 최적화까지** 왔습니다.
>
> 이 과정에서 배운 개별 기법들은 시간이 지나면 바뀔 것입니다.
> 새 모델이 나오고, 새 라이브러리가 나오고, 오늘의 최선이 내일은 구식이 될겁니다.
>
> 하지만 **바뀌지 않는 것**
>
> > **데이터를 먼저 보고, 가설을 세우고, 측정하고, 개선한다.**
>
> 이 태도가 이 과정에서 진짜로 남기고 싶었던 것입니다.
